<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/Qwen_3_TTS_By_(HunaarG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install with FlashAttention support
!pip install -U qwen-tts gradio huggingface_hub
!pip install flash-attn --no-build-isolation
!pip install pydub
!apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.3/113.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.3/566.3 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.4 MB/s eta 0:00:00
  Created wheel for sox: filename=sox-1.5.0-py3-none-any.whl size=40036 sha256=abd8d9

clone , custom , podcast với giọng mặc định

In [2]:
# import gradio as gr
# from qwen_tts import Qwen3TTSModel
# import torch, soundfile as sf, tempfile, gc
# import os

# # --- THÊM: Import thư viện xử lý âm thanh ---
# try:
#     from pydub import AudioSegment
# except ImportError:
#     print("⚠️ Chưa cài pydub. Vui lòng chạy '!pip install pydub' trước!")

# # ================= PERFORMANCE =================
# current_model = None
# current_model_type = None

# torch.backends.cudnn.benchmark = True
# torch.backends.cudnn.conv.fp32_precision = "tf32"
# torch.backends.cuda.matmul.fp32_precision = "tf32"

# print(f"🚀 GPU Detected: {torch.cuda.get_device_name(0)}")

# # ================= MODEL LOADER =================
# def load_model(t):
#     global current_model, current_model_type
#     if current_model_type == t:
#         return current_model

#     if current_model:
#         del current_model
#         gc.collect()
#         torch.cuda.empty_cache()

#     models = {
#         "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
#         "custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
#         "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
#     }

#     current_model = Qwen3TTSModel.from_pretrained(
#         models[t],
#         torch_dtype=torch.float16,
#         device_map="cuda:0",
#         attn_implementation="sdpa"
#     )
#     current_model_type = t
#     return current_model

# # ================= FUNCTIONS (GỐC) =================
# def voice_clone(text, audio, transcript, fast):
#     if not text or not audio: return None
#     m = load_model("base")
#     p = m.create_voice_clone_prompt(
#         ref_audio=audio,
#         ref_text=None if fast else transcript,
#         x_vector_only_mode=fast
#     )
#     with torch.inference_mode():
#         w, sr = m.generate_voice_clone(text=text, voice_clone_prompt=p)
#     f = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#     sf.write(f.name, w[0], sr)
#     return f.name

# def custom_voice(text, voice, inst):
#     if not text: return None
#     m = load_model("custom")
#     with torch.inference_mode():
#         w, sr = m.generate_custom_voice(
#             text=text,
#             speaker=voice,
#             instruct=inst if inst.strip() else None
#         )
#     f = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#     sf.write(f.name, w[0], sr)
#     return f.name

# def voice_design(text, desc):
#     if not text or not desc: return None
#     m = load_model("design")
#     with torch.inference_mode():
#         w, sr = m.generate_voice_design(text=text, instruct=desc)
#     f = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#     sf.write(f.name, w[0], sr)
#     return f.name

# # ================= FUNCTION MỚI: PODCAST MAKER =================
# def generate_podcast(script_text):
#     if not script_text: return None

#     # 1. Tải model Custom (Chỉ tải 1 lần)
#     m = load_model("custom")

#     # 2. Cấu hình nhân vật (Bạn có thể sửa Instruction ở đây)
#     CHAR_CONFIG = {
#         "Eric":   {"voice": "eric",   "inst": "Professional male teacher, deep warm voice, educational tone"},
#         "Serena": {"voice": "serena", "inst": "Friendly female host, energetic, happy tone"},
#         "Ryan":   {"voice": "ryan",   "inst": "British male voice, professional and calm"},
#         "Vivian": {"voice": "vivian", "inst": "Young female voice, fast and fun"}
#     }

#     final_audio = AudioSegment.silent(duration=500) # Bắt đầu với 0.5s im lặng
#     gap = AudioSegment.silent(duration=400) # Nghỉ 0.4s giữa các câu

#     lines = script_text.strip().split('\n')

#     # 3. Duyệt từng dòng kịch bản
#     for line in lines:
#         if ":" in line:
#             # Tách tên và lời thoại (VD: "Eric: Hello world")
#             name, text = line.split(":", 1)
#             name = name.strip()
#             text = text.strip()

#             # Kiểm tra xem tên có trong danh sách không
#             config = CHAR_CONFIG.get(name)

#             if config and text:
#                 print(f"🎙️ Đang tạo giọng cho {name}...")
#                 with torch.inference_mode():
#                     w, sr = m.generate_custom_voice(
#                         text=text,
#                         speaker=config["voice"],
#                         instruct=config["inst"]
#                     )

#                 # Lưu file tạm
#                 temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#                 sf.write(temp_wav.name, w[0], sr)

#                 # Đọc file bằng Pydub và ghép vào
#                 segment = AudioSegment.from_wav(temp_wav.name)
#                 final_audio += segment + gap

#                 # Xóa file tạm cho nhẹ máy
#                 os.unlink(temp_wav.name)
#             else:
#                 print(f"⚠️ Không tìm thấy cấu hình cho nhân vật: {name} (Hoặc lời thoại trống)")

#     # 4. Xuất file cuối cùng
#     output_path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
#     final_audio.export(output_path, format="mp3")
#     print("✅ Đã ghép xong Podcast!")
#     return output_path

# # ================= STRONG UI CSS =================
# css = """
# body{
# background:radial-gradient(circle at top,#eef2ff,#e0e7ff,#f8fafc);
# font-family:Inter,system-ui;
# }
# .hero{
# padding:26px;border-radius:20px;
# background:linear-gradient(135deg,#4f46e5,#7c3aed);
# color:white;text-align:center;
# box-shadow:0 20px 50px rgba(0,0,0,.25);
# margin-bottom:25px;
# }
# .hero h1{font-size:28px;margin-bottom:6px}
# .hero p{opacity:.9}
# .socials{display:flex;gap:14px;justify-content:center;flex-wrap:wrap;margin:22px 0}
# .btn{
# padding:12px 22px;border-radius:999px;
# font-weight:600;text-decoration:none;
# color:white;display:flex;gap:8px;
# align-items:center;transition:.35s;
# box-shadow:0 10px 30px rgba(0,0,0,.2)
# }
# .btn:hover{transform:translateY(-4px) scale(1.04)}
# .yt{background:#ff0000}
# .ig{background:linear-gradient(45deg,#f58529,#dd2a7b,#8134af)}
# .wa{background:#25D366}
# .panel{
# background:rgba(255,255,255,.7);
# backdrop-filter:blur(14px);
# border-radius:20px;
# padding:20px;
# box-shadow:0 15px 40px rgba(0,0,0,.15)
# }
# .footer{
# margin-top:30px;text-align:center;
# font-size:14px;color:#444
# }
# """

# # ================= UI =================
# with gr.Blocks(title="Qwen3-TTS | Hunaar Ansari", css=css) as demo:

#     gr.HTML("""
#     <div class="hero">
#         <h1>🎙️ Qwen3-TTS Voice Studio</h1>
#         <p>Professional AI Voice Cloning • Custom Voices • Voice Design</p>
#         <p><b>Created by Hunaar Ansari (Modified for Podcast)</b></p>
#     </div>
#     """)

#     gr.HTML("""
#     <div class="socials">
#         <a class="btn yt" href="http://www.youtube.com/@Hunaarg" target="_blank">YouTube</a>
#         <a class="btn ig" href="https://www.instagram.com/hunaar_ansari/" target="_blank">Instagram</a>
#         <a class="btn wa" href="https://whatsapp.com/channel/0029VabTEup4dTnKxHsYP01W" target="_blank">
#             WhatsApp Channel
#         </a>
#     </div>
#     """)

#     with gr.Tab("🎤 Voice Clone"):
#         with gr.Group(elem_classes="panel"):
#             t = gr.Textbox(lines=4, label="Text")
#             a = gr.Audio(type="filepath", label="Reference Audio")
#             tr = gr.Textbox(lines=2, label="Transcript (optional)")
#             f = gr.Checkbox(value=True, label="Fast Mode")
#             o = gr.Audio()
#             gr.Button("Generate Voice", variant="primary").click(
#                 voice_clone,[t,a,tr,f],o)

#     with gr.Tab("🎭 Custom Voice"):
#         with gr.Group(elem_classes="panel"):
#             t2 = gr.Textbox(lines=4)
#             v = gr.Dropdown(
#                 ["serena","vivian","ono_anna","sohee","aiden","dylan","eric","ryan","uncle_fu"],
#                 value="serena"
#             )
#             i = gr.Textbox(lines=2)
#             o2 = gr.Audio()
#             gr.Button("Generate Voice", variant="primary").click(
#                 custom_voice,[t2,v,i],o2)

#     with gr.Tab("🎨 Voice Design"):
#         with gr.Group(elem_classes="panel"):
#             t3 = gr.Textbox(lines=4)
#             d = gr.Textbox(lines=3)
#             o3 = gr.Audio()
#             gr.Button("Generate Voice", variant="primary").click(
#                 voice_design,[t3,d],o3)

#     # --- TAB MỚI: PODCAST MODE ---
#     with gr.Tab("🎬 Podcast Mode (Kịch bản)"):
#         with gr.Group(elem_classes="panel"):
#             gr.Markdown("### Nhập kịch bản theo mẫu: `Tên: Lời thoại`")
#             gr.Markdown("Nhân vật hỗ trợ: **Eric** (Nam), **Serena** (Nữ), **Ryan**, **Vivian**")

#             script_input = gr.Textbox(
#                 lines=10,
#                 label="Kịch bản (Script)",
#                 placeholder="Eric: Hello everyone, welcome to the show.\nSerena: Hi Eric! I am so happy to be here.\nEric: Today we talk about AI."
#             )
#             podcast_output = gr.Audio(label="File Podcast Hoàn Chỉnh")

#             gr.Button("🎬 TẠO PODCAST & GHÉP FILE", variant="primary").click(
#                 generate_podcast, [script_input], podcast_output
#             )

#     gr.HTML("""
#     <div class="footer">
#         🚀 Built by <b>Hunaar Ansari</b><br>
#         Follow • Join • Learn AI Voice Technology
#     </div>
#     """)

# print("🔥 Qwen3-TTS Ultra UI | Hunaar Ansari")
# demo.launch(share=True, debug=True, theme=gr.themes.Soft())

voice design podcast

In [5]:
# @title ⚡️ SPEED PODCAST STUDIO (Đã tối ưu bộ nhớ đệm)
import gradio as gr
from qwen_tts import Qwen3TTSModel
import torch, soundfile as sf, tempfile, gc
import os
import re

# --- Cài đặt thư viện ---
try:
    from pydub import AudioSegment
except ImportError:
    print("⏳ Đang cài pydub...")
    os.system('pip install pydub')
    os.system('apt-get install -y ffmpeg')
    from pydub import AudioSegment

# ================= QUẢN LÝ MODEL =================
torch.backends.cudnn.benchmark = True
current_model = None
current_model_type = None

def load_model_smart(task_type):
    global current_model, current_model_type

    if task_type == "DESIGN":
        model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
    else: # CLONE
        model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

    if current_model_type == task_type and current_model is not None:
        return current_model

    if current_model:
        del current_model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"📥 Đang tải Model {task_type}...")
    try:
        current_model = Qwen3TTSModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="cuda:0",
            attn_implementation="sdpa"
        )
        current_model_type = task_type
        print("✅ Model Ready!")
        return current_model
    except Exception as e:
        print(f"❌ Lỗi: {e}")
        return None

# ================= TAB 1: TẠO MẪU (Giữ nguyên) =================
def create_voice_sample(prompt_text, sample_text):
    model = load_model_smart("DESIGN")
    if not model: return None

    print(f"🎨 Đang vẽ giọng...")
    with torch.inference_mode():
        w, sr = model.generate_voice_design(text=sample_text, instruct=prompt_text)

    temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav").name
    sf.write(temp_wav, w[0], sr)
    return temp_wav

# ================= TAB 2: SẢN XUẤT SIÊU TỐC (NEW) =================
def generate_podcast_fast(script_text, nam_ref_path, nu_ref_path):
    if not script_text or not nam_ref_path or not nu_ref_path:
        return None

    model = load_model_smart("CLONE")

    final_audio = AudioSegment.silent(duration=500)
    gap = AudioSegment.silent(duration=580)

    lines = script_text.strip().split('\n')
    total_lines = len([l for l in lines if ":" in l])

    # --- BƯỚC CẢI TIẾN: TÍNH TOÁN TRƯỚC (PRE-CALCULATE) ---
    print("⚡️ Đang phân tích file mẫu giọng Nam & Nữ (Chỉ làm 1 lần)...")

    try:
        # Tính toán "Bản đồ giọng" cho Nam
        nam_prompt_feature = model.create_voice_clone_prompt(
            ref_audio=nam_ref_path, ref_text=None, x_vector_only_mode=True
        )
        # Tính toán "Bản đồ giọng" cho Nữ
        nu_prompt_feature = model.create_voice_clone_prompt(
            ref_audio=nu_ref_path, ref_text=None, x_vector_only_mode=True
        )
        print("✅ Đã ghi nhớ giọng! Bắt đầu render siêu tốc...")
    except Exception as e:
        print(f"❌ Lỗi đọc file âm thanh mẫu: {e}")
        return None

    # --- VÒNG LẶP (Giờ chỉ việc lấy ra dùng) ---
    for index, line in enumerate(lines):
        if ":" in line:
            name_part, text = line.split(":", 1)
            name_part = name_part.strip()
            text = text.strip()

            clean_name = re.sub(r'\(.*?\)', '', name_part).strip().lower()
            is_nam = clean_name in ["nam", "man", "male", "host", "teacher", "eric", "ryan"]

            # Lấy bản đồ giọng đã tính sẵn
            current_prompt = nam_prompt_feature if is_nam else nu_prompt_feature

            if text:
                print(f"🎙️ [{index+1}/{total_lines}] {clean_name.upper()}...")

                with torch.inference_mode():
                    w, sr = model.generate_voice_clone(
                        text=text,
                        voice_clone_prompt=current_prompt # <--- Dùng lại biến này, cực nhanh
                    )

                temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
                sf.write(temp_wav.name, w[0], sr)

                segment = AudioSegment.from_wav(temp_wav.name)
                final_audio += segment + gap
                os.unlink(temp_wav.name)

    output_path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
    final_audio.export(output_path, format="mp3")
    print("✅ HOÀN TẤT!")
    return output_path

# ================= GIAO DIỆN =================
css = """
body{background: #0b0f19; color: #e5e7eb;}
.panel{background: #1f2937; padding: 20px; border-radius: 12px;}
"""

with gr.Blocks(title="Speed Podcast Studio", css=css, theme=gr.themes.Base()) as demo:
    gr.Markdown("# ⚡️ SPEED PODCAST STUDIO (Bản Tối Ưu Tốc Độ)")

    with gr.Tab("B1: TẠO MẪU"):
        with gr.Row():
            with gr.Column(elem_classes="panel"):
                gr.Markdown("### 👨 Nam")
                nam_p = gr.Textbox(label="Prompt Nam", value="A professional male podcast host, deep and soothing voice. American accent.")
                nam_s = gr.Textbox(label="Text mẫu", value="This is my voice.")
                b_nam = gr.Button("Tạo Mẫu Nam")
                o_nam = gr.Audio(type="filepath")
            with gr.Column(elem_classes="panel"):
                gr.Markdown("### 👩 Nữ")
                nu_p = gr.Textbox(label="Prompt Nữ", value="A warm female voice, soft and velvety. American accent.")
                nu_s = gr.Textbox(label="Text mẫu", value="This is my voice.")
                b_nu = gr.Button("Tạo Mẫu Nữ")
                o_nu = gr.Audio(type="filepath")

    with gr.Tab("B2: RENDER NHANH"):
        with gr.Row():
            with gr.Column(elem_classes="panel"):
                r_nam = gr.Audio(label="File Nam", type="filepath")
                r_nu = gr.Audio(label="File Nữ", type="filepath")
            with gr.Column(elem_classes="panel"):
                script = gr.Textbox(lines=10, label="Script")
                btn = gr.Button("🚀 RENDER SIÊU TỐC", variant="primary")
                out = gr.Audio(label="Kết quả")

    b_nam.click(create_voice_sample, [nam_p, nam_s], [o_nam])
    b_nu.click(create_voice_sample, [nu_p, nu_s], [o_nu])
    o_nam.change(lambda x: x, o_nam, r_nam)
    o_nu.change(lambda x: x, o_nu, r_nu)
    btn.click(generate_podcast_fast, [script, r_nam, r_nu], out)

demo.launch(share=True, debug=True)

/tmp/ipython-input-2561655115.py:136: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Speed Podcast Studio", css=css, theme=gr.themes.Base()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ee1d1db5aa006dcc9e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📥 Đang tải Model CLONE...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model Ready!
⚡️ Đang phân tích file mẫu giọng Nam & Nữ (Chỉ làm 1 lần)...


...

🎙️ [17/19] NAM...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


🎙️ [18/19] NU...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


🎙️ [19/19] NAM...
✅ HOÀN TẤT!
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ee1d1db5aa006dcc9e.gradio.live
